# Taxi IA Chapinero · Análisis exploratorio de la red OSM integrada

**Tercer corte · Preparación previa al aprendizaje automático**

Este notebook describe el **dataset operativo único** `dataset_red_osm_operativo_auditado.csv.gz`, generado a partir de las aristas dirigidas OSM y patrones históricos de Bitcarrier de **mayo de 2022**. Cada fila representa una **arista OSM × día tipo × hora**. Las velocidades son **estimadas**: transferencia histórica **provisional** o **simulación calibrada**; ninguna es una medición directa de esa arista.

**Preguntas del EDA:** cobertura y unicidad, procedencia, distribución de velocidad y tiempo, patrones por hora/día, relación con longitud y clase de vía, y limitaciones para entrenar y evaluar modelos.

**Importante:** el análisis exploratorio caracteriza la matriz operativa, pero **no constituye prueba de desempeño predictivo real**. El histórico de Bitcarrier reservado se utilizará para evaluar posteriormente la regresión. Las unidades fueron verificadas: distancia en metros y velocidad en km/h.

## 1. Librerías y carga reproducible

El archivo se genera previamente ejecutando `construir_red_osm_integrada.py`. Se puede abrir el notebook desde VS Code con el kernel Python del backend. `EDA_RED_OSM_INPUT` permite cambiar la ruta únicamente para reproducibilidad de pruebas.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DIAS = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
raiz = Path.cwd()
if not (raiz / 'backend').exists():
    raiz = raiz.parents[1] if raiz.name == 'notebooks' else raiz
archivo = Path(os.environ.get('EDA_RED_OSM_INPUT', str(raiz / 'backend/datos/ml/red_osm_auditada/dataset_red_osm_operativo_auditado.csv.gz')))
assert archivo.exists(), f'Falta: {archivo}. Ejecuta auditar_y_cerrar_red_osm.py desde la raíz.'
df = pd.read_csv(archivo)
print(f'Filas: {len(df):,} | Aristas: {df.indice_arista_osm.nunique():,} | Días: {df.dia_semana.nunique()} | Horas: {df.hora.nunique()}')
print('Columnas:', ', '.join(df.columns))
display(df.head(5))

## 2. Integridad, granularidad y consistencia física

Verificar que no haya pares repetidos `(arista, día, hora)`, que cada arista tenga exactamente 168 combinaciones y que las unidades asumidas produzcan tiempos coherentes.

In [ ]:
claves = ['indice_arista_osm','dia_semana','hora']
conteos = df.groupby('indice_arista_osm').size()
inconsistencias = ((.06*df.distancia_m/df.velocidad_kmh - df.tiempo_min).abs() > .00015).sum()
calidad = pd.DataFrame({'indicador':['Duplicados arista/día/hora','Aristas con cobertura != 168','Distancia no positiva','Velocidad no positiva','Tiempo no positivo','Tiempo inconsistente con fórmula'],
                        'cantidad':[df.duplicated(claves).sum(),(conteos!=168).sum(),(df.distancia_m<=0).sum(),(df.velocidad_kmh<=0).sum(),(df.tiempo_min<=0).sum(),inconsistencias]})
display(calidad)
assert calidad.cantidad.eq(0).all(), 'Corrige fallos de calidad antes de entrenar modelos'
print('Cobertura completa: cada arista aparece 7 días × 24 horas.')

## 3. Procedencia de las estimaciones

El archivo operativo es **uno solo**, pero se conservan etiquetas para no confundir transferencias provisionales con simulación. El número de filas no significa un número equivalente de mediciones independientes.

In [ ]:
conteo_fuentes = df.fuente_dato.value_counts().rename_axis('fuente').reset_index(name='escenarios')
conteo_fuentes['porcentaje'] = (100*conteo_fuentes.escenarios/len(df)).round(2)
display(conteo_fuentes)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(conteo_fuentes.fuente,conteo_fuentes.escenarios)
ax.set_ylabel('Cantidad de escenarios')
ax.set_title('Composición del dataset operativo por procedencia')
ax.tick_params(axis='x',labelrotation=12)
fig.tight_layout(); plt.show()
tr = df.loc[df.fuente_dato.eq('TRANSFERENCIA_BITCARRIER_PROVISIONAL')]
print('Aristas con al menos un horario transferido:',tr.indice_arista_osm.nunique())
print('Aristas con cobertura transferida parcial:',int((tr.groupby('indice_arista_osm').size()<168).sum()))

## 4. Distribuciones y valores extremos

Explorar los rangos sin eliminar automáticamente valores atípicos. Como el tiempo depende de la distancia, una arista larga puede tener un tiempo mayor sin representar necesariamente alta congestión.

In [ ]:
display(df[['distancia_m','velocidad_kmh','tiempo_min']].describe(percentiles=[.25,.5,.75,.95]).T)
print('Escenarios con velocidad > 100 (km/h):',int((df.velocidad_kmh>100).sum()))
fig,ax=plt.subplots(figsize=(9,4))
ax.hist(df.velocidad_kmh,bins=45)
ax.set(title='Distribución de velocidad estimada',xlabel='Velocidad (km/h)',ylabel='Escenarios')
fig.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
ax.hist(df.tiempo_min,bins=55)
ax.set(title='Distribución de tiempo de desplazamiento derivado',xlabel='Tiempo estimado (minutos)',ylabel='Escenarios')
fig.tight_layout();plt.show()

## 5. Perfil horario y diferencias por día

Estas son **horas tipo** de una semana modelada, no un registro continuo de tráfico de 2026.

In [ ]:
p=df.groupby('hora').velocidad_kmh.median()
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(p.index,p.values,marker='o')
ax.set(title='Velocidad mediana estimada por hora tipo',xlabel='Hora',ylabel='Velocidad estimada (km/h)',xticks=range(0,24,2))
ax.grid(alpha=.2);fig.tight_layout();plt.show()

In [ ]:
calor=df.pivot_table(index='dia_semana',columns='hora',values='velocidad_kmh',aggfunc='median').reindex(index=range(7),columns=range(24))
fig,ax=plt.subplots(figsize=(11,4))
im=ax.imshow(calor.values,aspect='auto')
ax.set(yticks=range(7),yticklabels=DIAS,xticks=range(0,24,2),xticklabels=[str(x) for x in range(0,24,2)],title='Mapa de calor de velocidad mediana · día × hora',xlabel='Hora tipo')
fig.colorbar(im,ax=ax,label='Velocidad estimada (km/h)')
fig.tight_layout();plt.show()

## 6. Características de las vías y tiempo

La relación distancia–tiempo requiere una precaución: el tiempo objetivo se calcula usando precisamente distancia y velocidad. **No se puede usar velocidad del mismo escenario como predictor** del tiempo derivado; produciría una fuga de información.

In [ ]:
car=df.groupby('indice_arista_osm').agg(distancia_m=('distancia_m','first'),tiempo_mediano=('tiempo_min','median'),velocidad_mediana=('velocidad_kmh','median')).reset_index()
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(car.distancia_m,car.tiempo_mediano,s=9,alpha=.42)
ax.set(title='Distancia vs tiempo mediano por arista',xlabel='Distancia (m)',ylabel='Tiempo mediano derivado (min)')
fig.tight_layout();plt.show()
print('Correlación entre distancia y tiempo mediano por arista:',round(car.distancia_m.corr(car.tiempo_mediano),3))

In [ ]:
tipos = (df.groupby('tipo_via',dropna=False).agg(velocidad_mediana=('velocidad_kmh','median'),tiempo_mediano=('tiempo_min','median'),aristas=('indice_arista_osm','nunique')).sort_values('aristas',ascending=False).head(12))
display(tipos)
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(tipos.index.astype(str),tipos.velocidad_mediana)
ax.set(title='Velocidad mediana estimada por clase de vía',ylabel='Velocidad estimada (km/h)')
ax.tick_params(axis='x',labelrotation=35)
fig.tight_layout();plt.show()

## 7. Conclusiones y decisiones para Machine Learning

1. **Unidad operativa común:** el dataset describe 1.741 aristas dirigidas con 168 perfiles semanales y cero combinaciones duplicadas, siempre que pasen las comprobaciones anteriores.
2. **Procedencia:** la red se completa con simulación calibrada y transferencias Bitcarrier **provisionales**; **ninguna** representa medición directa de tráfico para esa arista.
3. **EDA y entrenamiento:** para describir las condiciones operativas, usar toda la matriz OSM integrada; para evaluar generalización a movilidad observada, usar las observaciones Bitcarrier reservadas (no la simulación).
4. **Regresión:** variable objetivo `tiempo_min` (derivada, no observada directamente). Predictores candidatos: `distancia_m`, `hora`, `dia_semana`, `tipo_via` y atributos disponibles del grafo. Excluir `velocidad_kmh` del mismo escenario y cualquier variable derivada de la etiqueta.
5. **Clasificación:** definir explícitamente categorías baja/media/alta y su justificación. Evaluar en datos históricos independientes si las etiquetas derivan de velocidad observada.
6. **Agrupamiento:** agrupar perfiles de aristas (una observación por arista con agregados horarios), no cada fila semanal como si fuera un segmento geográfico diferente.
7. **Pendientes críticos:** mantener documentadas las correspondencias automáticas provisionales y tomar la direccionalidad del grafo OSM, y documentar que los patrones corresponden a una semana tipo calibrada con datos de 2022.

El JSON del dashboard se genera con `backend/scripts/ml/generar_eda_red_osm.py` y usa **exactamente el mismo CSV**.